# 🎬 Análisis de Sentimientos en Reseñas de Películas

Pipeline completo de NLP para clasificar reseñas de películas como positivas o negativas.
Se comparan dos representaciones: **Bag of Words** y **TF-IDF**, ambas con Regresión Logística.

**Dataset:** IMDB — 50.000 reseñas balanceadas  
**Stack:** Python · pandas · NLTK · scikit-learn

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 2. Carga del dataset

In [5]:
df=pd.read_csv("/content/IMDB-Dataset.csv", engine='python', on_bad_lines='skip')

display(df) #Forma mas linda de ver los datos que con el print.

#Muestra de cantidad de documentos y clases

print("La cantidad de documentos son: ",df.shape[0])

print("La cantidad de clases son: ",df.shape[1])

#Muestra de distribucion de cada clase
df['review'].value_counts()
print()
df['sentiment'].value_counts() #Podemos observar que hay un total de 50.000 documentos los cuales se clasifican en sentimiento positivo o negativo, y estan perfectamente distribuidos en 25,000 para cada opcion.

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


La cantidad de documentos son:  50000
La cantidad de clases son:  2



,count
sentiment,
positive,25000
negative,25000


## 3. Tokenización, Stemming y Lematización

Se aplican tres técnicas sobre una reseña de ejemplo:
- **Tokenización:** divide el texto en palabras individuales
- **Stemming:** reduce cada palabra a su raíz (*watching* → *watch*)
- **Lematización:** convierte a forma base canónica, gramaticalmente correcta

In [8]:
reseña_elegida=(df['review'][2]) #Decido elejir la reseña 3 para trabajar.
print(reseña_elegida)
token=word_tokenize(reseña_elegida)
display("Proceso de tokenizazion:  ")
print(token)
#Resumen= Aplico tokenization a la reseña numero 2 del csv y imprimo en pantalla sus caracteres del 0 al 60

# Aplico Stemming
stemmer=PorterStemmer()
display("Proceso de stemming:  ")
steams=[stemmer.stem(tokens) for tokens in token]
print(steams)

#Lemattizacion
Lemmatizer=WordNetLemmatizer()
Lemma=[Lemmatizer.lemmatize(tokens) for tokens in token]
display("Proceso de lemmatizacion:  ")
print(Lemma)




I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). While some may be disappointed when they realize this is not Match Point 2: Risk Addiction, I thought it was proof that Woody Allen is still fully in control of the style many of us have grown to love.<br /><br />This was the most I'd laughed at one of Woody's comedies in years (dare I say a decade?). While I've never been impressed with Scarlet Johanson, in this she managed to tone down her "sexy" image and jumped right into a average, but spirited young woman.<br /><br />This may not be the crown jewel of his career, but it was wittier than "Devil Wears Prada" and more interesting than "Superman" a great comedy to go see with friends.


'Proceso de tokenizazion:  '

['I', 'thought', 'this', 'was', 'a', 'wonderful', 'way', 'to', 'spend', 'time', 'on', 'a', 'too', 'hot', 'summer', 'weekend', ',', 'sitting', 'in', 'the', 'air', 'conditioned', 'theater', 'and', 'watching', 'a', 'light-hearted', 'comedy', '.', 'The', 'plot', 'is', 'simplistic', ',', 'but', 'the', 'dialogue', 'is', 'witty', 'and', 'the', 'characters', 'are', 'likable', '(', 'even', 'the', 'well', 'bread', 'suspected', 'serial', 'killer', ')', '.', 'While', 'some', 'may', 'be', 'disappointed', 'when', 'they', 'realize', 'this', 'is', 'not', 'Match', 'Point', '2', ':', 'Risk', 'Addiction', ',', 'I', 'thought', 'it', 'was', 'proof', 'that', 'Woody', 'Allen', 'is', 'still', 'fully', 'in', 'control', 'of', 'the', 'style', 'many', 'of', 'us', 'have', 'grown', 'to', 'love.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'This', 'was', 'the', 'most', 'I', "'d", 'laughed', 'at', 'one', 'of', 'Woody', "'s", 'comedies', 'in', 'years', '(', 'dare', 'I', 'say', 'a', 'decade', '?', ')', '.', 'While', 'I'

'Proceso de stemming:  '

['i', 'thought', 'thi', 'wa', 'a', 'wonder', 'way', 'to', 'spend', 'time', 'on', 'a', 'too', 'hot', 'summer', 'weekend', ',', 'sit', 'in', 'the', 'air', 'condit', 'theater', 'and', 'watch', 'a', 'light-heart', 'comedi', '.', 'the', 'plot', 'is', 'simplist', ',', 'but', 'the', 'dialogu', 'is', 'witti', 'and', 'the', 'charact', 'are', 'likabl', '(', 'even', 'the', 'well', 'bread', 'suspect', 'serial', 'killer', ')', '.', 'while', 'some', 'may', 'be', 'disappoint', 'when', 'they', 'realiz', 'thi', 'is', 'not', 'match', 'point', '2', ':', 'risk', 'addict', ',', 'i', 'thought', 'it', 'wa', 'proof', 'that', 'woodi', 'allen', 'is', 'still', 'fulli', 'in', 'control', 'of', 'the', 'style', 'mani', 'of', 'us', 'have', 'grown', 'to', 'love.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'thi', 'wa', 'the', 'most', 'i', "'d", 'laugh', 'at', 'one', 'of', 'woodi', "'s", 'comedi', 'in', 'year', '(', 'dare', 'i', 'say', 'a', 'decad', '?', ')', '.', 'while', 'i', "'ve", 'never', 'been', 'impress', 'with',

'Proceso de lemmatizacion:  '

['I', 'thought', 'this', 'wa', 'a', 'wonderful', 'way', 'to', 'spend', 'time', 'on', 'a', 'too', 'hot', 'summer', 'weekend', ',', 'sitting', 'in', 'the', 'air', 'conditioned', 'theater', 'and', 'watching', 'a', 'light-hearted', 'comedy', '.', 'The', 'plot', 'is', 'simplistic', ',', 'but', 'the', 'dialogue', 'is', 'witty', 'and', 'the', 'character', 'are', 'likable', '(', 'even', 'the', 'well', 'bread', 'suspected', 'serial', 'killer', ')', '.', 'While', 'some', 'may', 'be', 'disappointed', 'when', 'they', 'realize', 'this', 'is', 'not', 'Match', 'Point', '2', ':', 'Risk', 'Addiction', ',', 'I', 'thought', 'it', 'wa', 'proof', 'that', 'Woody', 'Allen', 'is', 'still', 'fully', 'in', 'control', 'of', 'the', 'style', 'many', 'of', 'u', 'have', 'grown', 'to', 'love.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'This', 'wa', 'the', 'most', 'I', "'d", 'laughed', 'at', 'one', 'of', 'Woody', "'s", 'comedy', 'in', 'year', '(', 'dare', 'I', 'say', 'a', 'decade', '?', ')', '.', 'While', 'I', "'ve",

## 4. Preprocesamiento completo del corpus

Pipeline aplicado sobre las 50.000 reseñas:
1. Conversión a minúsculas
2. Tokenización
3. Eliminación de stopwords
4. Lematización

In [9]:
reseñas_full=(df["review"])#Reseñas full contiene todo el corpus del documento.
#1.Cambio a miniscula
reseñas_full=reseñas_full.str.lower()
#2.Tokenizacion
reseñas_tokenizadas = reseñas_full.apply(word_tokenize)
#Eliminacion de stopwards
stop_words=set(stopwords.words('english'))
reseñas_sin_stopwords = reseñas_tokenizadas.apply(lambda x: [word for word in x if word not in stop_words])#
#Lemmatizacion
Lemmatizer=WordNetLemmatizer()
reseñas_lematizadas = reseñas_sin_stopwords.apply(lambda x: [Lemmatizer.lemmatize(word) for word in x])


documento_original=df['review']
documento_procesado=reseñas_lematizadas
print("Documento original n°1:  ")
print(documento_original[0])
print("Documento original n°2:  ")
print(documento_original[1])
print("Documento original n°3:  ")
print(documento_original[2])
print("Documento procesado n°1:  ")
print(documento_procesado[0])
print("Documento procesado n°2:  ")
print(documento_procesado[1])
print("Documento procesado n°3:  ")
print(documento_procesado[2])

Documento original n°1:  
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of 

## 5. Representación Bag of Words

Se construye la matriz de frecuencias con CountVectorizer.
La matriz es **dispersa (sparse)** porque cada documento usa solo una fracción del vocabulario total.

In [11]:

corpus = documento_procesado.apply(lambda x: ' '.join(x))

# REPRESENTACION DE BAG OF WORDS
vectorizer = CountVectorizer()
bags_of_words = vectorizer.fit_transform(corpus)
print("Representacion de Bag of Words:")
print(bags_of_words)

# 1. Tamaño del vocabulario
vocabulary_tamaño = len(vectorizer.get_feature_names_out())
print(f"Tamaño del vocabulario: {vocabulary_tamaño}\n")

# 2. Las 20 palabras más frecuentes
#OBTENER NOMBRES DE ESAS PALABRAS:
palabras=vectorizer.get_feature_names_out()
#SUMAMOS FRECUENCIA DE CADA PALABRA:
frecuencias=bags_of_words.sum(axis=0).A1


#CREAMOS DATAFRAME:
df2=pd.DataFrame({
    'palabra': palabras,
    'frecuencia': frecuencias
})
print("Las 20 palabras mas frecuentes son :  ")
print(df2.sort_values('frecuencia',ascending=False).head(20))




# 3. Dimensión de la matriz resultante
matrix_dimensions = bags_of_words.shape
print(f"Dimensión de la matriz resultante: {matrix_dimensions}")

# Análisis: ¿Por qué la matriz resultante es dispersa (sparse)?
#en la mayoría de las posiciones de la matriz hay 0, porque cada documento usa solo una pequeña parte del vocabulario


Representacion de Bag of Words:
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4875695 stored elements and shape (50000, 95369)>
  Coords	Values
  (0, 60187)	1
  (0, 70710)	1
  (0, 53998)	1
  (0, 91886)	2
  (0, 61450)	6
  (0, 28218)	2
  (0, 49889)	3
  (0, 40089)	1
  (0, 71086)	2
  (0, 29038)	1
  (0, 37848)	1
  (0, 53450)	1
  (0, 11410)	6
  (0, 31406)	2
  (0, 84529)	1
  (0, 80984)	2
  (0, 12277)	1
  (0, 88460)	1
  (0, 73737)	1
  (0, 90881)	4
  (0, 75195)	1
  (0, 93635)	2
  (0, 35375)	2
  (0, 86887)	1
  (0, 76251)	4
  :	:
  (49999, 73486)	1
  (49999, 46989)	1
  (49999, 88497)	1
  (49999, 48766)	1
  (49999, 43087)	1
  (49999, 29350)	1
  (49999, 6855)	1
  (49999, 37931)	1
  (49999, 42039)	1
  (49999, 41734)	1
  (49999, 5794)	1
  (49999, 58576)	1
  (49999, 86425)	1
  (49999, 46683)	1
  (49999, 79342)	1
  (49999, 53181)	1
  (49999, 69969)	1
  (49999, 13092)	1
  (49999, 15023)	1
  (49999, 29362)	1
  (49999, 35708)	1
  (49999, 69963)	1
  (49999, 56634)	1
  (49999, 19947)	1
  (4999

## 6. División Train/Test

80% entrenamiento / 20% prueba.
Evaluar con los mismos datos de entrenamiento inflaría las métricas artificialmente.

In [12]:
x = df['review']
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

#Respuesta a pregunta= Si se evalua un modelo con los mismo datos que se entrena no tendria sentido ya que para evaluar el modelo es necesario meter un dato el cual no haya estado en el entrenamiento para ver como responde el modelo por si solo y que prediccion da




## 7. Modelo con Bag of Words — Regresión Logística

In [13]:
vectorizer = CountVectorizer()

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

modelo = LogisticRegression()
modelo.fit(x_train_vec, y_train)



#CALCULAR ACCURY,PRECISION,RECALL,F1 (SCORE)(PRUEBA)
y_pred = modelo.predict(x_test_vec)
accuracy = accuracy_score(y_test, y_pred)
precision=precision_score(y_test,y_pred,pos_label='positive')
recall=recall_score(y_test,y_pred,pos_label='positive')
f1=f1_score(y_test,y_pred,pos_label='positive')

print("Accuracy de prueba:", accuracy)
print("Precision de prueba:", precision)
print("Recall de prueba:", recall)
print("F1-score prueba :", f1)
print()


#CALCULAR ACCURACY PARA DATOS DE ENTRENAMIENTO
y_pred_train2=modelo.predict(x_train_vec)
accuracy_train2=accuracy_score(y_train,y_pred_train2)
precision_train2=precision_score(y_train,y_pred_train2,pos_label='positive')
recall_train2=recall_score(y_train,y_pred_train2,pos_label='positive')
f1_train2=f1_score(y_train,y_pred_train2,pos_label='positive')
print("Accuracy de entrenamiento:", accuracy_train2)
print("Precision de entrenamiento:", precision_train2)
print("Recall de entrenamiento:", recall_train2)
print("F1-score entrenamiento :", f1_train2)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Accuracy de prueba: 0.8916
Precision de prueba: 0.886305919124829
Recall de prueba: 0.9003770589402659
F1-score prueba : 0.8932860799369955

Accuracy de entrenamiento: 0.957325
Precision de entrenamiento: 0.9553028035518308
Recall de entrenamiento: 0.9593707730073644
F1-score entrenamiento : 0.9573324668183067


## 8. Matriz de Confusión

El modelo clasificó correctamente ~90% de sentimientos positivos y ~88% de negativos.

In [14]:
matriz = confusion_matrix(y_test, y_pred)

df_matriz = pd.DataFrame(
    matriz,
    index=["Real Negativo", "Real Positivo"],
    columns=["Predicho Negativo", "Predicho Positivo"]
)

print(df_matriz)
#Respuesta= Se puede decir que el modelo predijo un poco mejor los reales positivos,alrededor de un 90%,mientras que los reales negativos le fue un poco peor, 88%.
#ConfusionMatrixDisplay.from_predictions(y_test,y_pred)

               Predicho Negativo  Predicho Positivo
Real Negativo               4379                582
Real Positivo                502               4537


## 9. Palabras más importantes para la clasificación

- **Positivas:** *refreshing, subtle, superb, perfectly, surprisingly...*
- **Negativas:** *waste, disappointment, poorly, lacks, disappointing...*

In [15]:
coeficientes= pd.Series(
    modelo.coef_[0],
    index=vectorizer.get_feature_names_out()
)

print("Las 10 palabras con coeficientes mas positivos fueron:  ")
print(coeficientes.sort_values(ascending=False).head(10))

print("Las 10 palabras con coeficientes mas negativos fueron:  ")
print(coeficientes.sort_values(ascending=True).head(10))


Las 10 palabras con coeficientes mas positivos fueron:  
refreshing      1.485651
subtle          1.323991
superb          1.277803
perfectly       1.220302
surprisingly    1.212339
enjoyable       1.199047
surprised       1.198151
gem             1.196377
funniest        1.192913
wonderfully     1.192701
dtype: float64
Las 10 palabras con coeficientes mas negativos fueron:  
waste            -2.566109
disappointment   -2.032661
poorly           -1.839925
lacks            -1.816784
disappointing    -1.661411
dull             -1.523683
avoid            -1.519572
awful            -1.489672
fails            -1.481093
lame             -1.472192
dtype: float64


## 10. Modelo con TF-IDF — Comparación

TF-IDF pondera palabras según frecuencia en el documento y rareza en el corpus.
Obtuvo mejores resultados que Bag of Words en todas las métricas.

In [16]:

tfid= TfidfVectorizer()

x_train_tfid=tfid.fit_transform(x_train)
x_test_tfid=tfid.transform(x_test)

modelo2=LogisticRegression()
modelo2.fit(x_train_tfid,y_train)


#METRICAS PRUEBA
y_pred2=modelo2.predict(x_test_tfid)

accuracy2=accuracy_score(y_test,y_pred2)
precision2=precision_score(y_test,y_pred2,pos_label='positive')
recall2=recall_score(y_test,y_pred2,pos_label='positive')
f12=f1_score(y_test,y_pred2,pos_label='positive')

print("Accuracy de prueba:", accuracy2)
print("Precision de prueba:", precision2)
print("Recall de prueba :", recall2)
print("F1-score de prueba :", f12)
print()
#METRICAS ENTRENAMIENTO
y_pred_train=modelo2.predict(x_train_tfid)

accuracy_train=accuracy_score(y_train,y_pred_train)
preccision_train=precision_score(y_train,y_pred_train,pos_label='positive')
recall_train=recall_score(y_train,y_pred_train,pos_label='positive')
f1_train=f1_score(y_train,y_pred_train,pos_label='positive')

print("Accuracy de entrenamiento:", accuracy_train)
print("Precision de entrenamiento:", preccision_train)
print("Recall de entrenamiento:", recall_train)
print("F1-score de entrenamiento:", f1_train)

print()

#MATRIZ DE CONFUSION 2
matriz2=confusion_matrix(y_test,y_pred2)

df_matriz2=pd.DataFrame(
    matriz2,
    index=["Real negativo","Real positivo"],
    columns=["Prediccion negativo","Prediccion positivo"]
)

print(df_matriz2)

#MEJORES RESULTADOS:
# A traves de las metricas vistas sumando a una nueva matriz de confusion se puede ver que este modelo realizas unas breves mejores predicciones

Accuracy de prueba: 0.8999
Precision de prueba: 0.8914307871267934
Recall de prueba : 0.9124826354435404
F1-score de prueba : 0.9018338727076591

Accuracy de entrenamiento: 0.931025
Precision de entrenamiento: 0.9252447345001483
Recall de entrenamiento: 0.9375281799509043
F1-score de entrenamiento: 0.9313459576479957

               Prediccion negativo  Prediccion positivo
Real negativo                 4401                  560
Real positivo                  441                 4598


## 11. Conclusiones

- **Mejor representación:** TF-IDF superó a Bag of Words en todas las métricas
- **Overfitting:** No detectado — métricas de train y test similares
- **Mejoras propuestas:** Bigramas, ajuste de min_df/max_df en TF-IDF, explorar SVM o Transformers